# 03_results.ipynb - Tổng hợp kết quả, ablation và phân tích lỗi

**Người phụ trách:** An (nhóm trưởng) &nbsp;|&nbsp; **Việc:** A-04, A-05, A-10 &nbsp;|&nbsp; **Dùng cho:** báo cáo Chương 4 (mục 4.3, 4.4, 4.6, 4.8), slide 11-14

**Đầu vào:** `results/exp{id}_fold{k}.json` của EXP1-EXP5 (mỗi người trong nhóm chạy bằng `02_train.ipynb`), cache log-Mel `cache/logmel{64|128}.pt`, `meta/esc50.csv`, `meta/esc50-human.xlsx` (bonus).

**Đầu ra (lưu vào `figures/` và `tables/`):**

| File | Dùng cho |
|---|---|
| `tab_results.csv` | Bảng kết quả EXP1-EXP5 (mục 4.3) |
| `tab_rq.csv` | Bảng RQ -> cặp EXP -> chênh lệch -> p-value (mục 4.4) |
| `tab_gap_train_val.csv`, `tab_cost.csv` | Giả thuyết H2 (SpecAug) và chi phí tính toán (mục 4.5) |
| `fig6_ablation_bar.png` | Biểu đồ cột có error bar |
| `fig6_confmat_best.png`, `fig6_f1_per_class.png` | Confusion matrix, F1 theo lớp (mục 4.6) |
| `fig6_errors_pair1..5.png`, `tab_confused_pairs.csv` | Ví dụ lỗi cụ thể (mục 4.6) |
| `tab_nghe_lai.csv` | Danh sách clip để nghe lại và phân loại lỗi do mô hình hay do dữ liệu |

**Cách chạy**
1. Lần đầu: để `USE_FAKE = True` để chạy thử cả notebook bằng dữ liệu **giả** (file xuất ra có tiền tố `fake_` và hình có chữ đỏ "DỮ LIỆU GIẢ"). Không dùng các file này trong báo cáo.
2. Khi đủ kết quả thật của cả nhóm: đổi `USE_FAKE = False`, chạy lại từ đầu (Runtime -> Run all).
3. Copy các file trong `figures/` và `tables/` vào repo rồi commit lên nhánh `an-eval`.

**Quy ước:** Accuracy và Macro-F1 tính theo %, mean ± std qua 5 fold (std mẫu, ddof=1); test chỉ dùng để báo cáo, không dùng để chọn mô hình.

## 0. Tham số và thiết lập môi trường

In [ ]:
# ===== THAM SỐ (chỉ sửa ở ô này) =====
USE_FAKE = True            # True: test bằng dữ liệu GIẢ | False: dùng kết quả thật của nhóm
EXP_IDS = [1, 2, 3, 4, 5]  # thêm 6 nếu nhóm có chạy EXP6
BEST_EXP = None            # None: tự chọn theo validation | hoặc gán tay, ví dụ BEST_EXP = 5
REPO_URL = "https://github.com/<ten-nhom>/<ten-repo>.git"   # chỉ cần khi chạy trên Colab
DRIVE_ROOT = "/content/drive/MyDrive/esc50"                    # thư mục Drive dùng chung của nhóm
REPO_SUBDIR = "DOANCUOIKY/ESC50"   # thư mục dự án nằm trong repo (chứa src/, configs/...); để "" nếu src/ nằm ở gốc repo

In [ ]:
import os, sys, zlib, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=False)
    os.chdir(os.path.join("/content/repo", REPO_SUBDIR))
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src import eval_lib as ev

if USE_FAKE:
    RESULT_DIR = "tests/fake_results"
elif IN_COLAB:
    RESULT_DIR = f"{DRIVE_ROOT}/results"
else:
    RESULT_DIR = "results"

if IN_COLAB:
    META_CSV   = f"{DRIVE_ROOT}/data/ESC-50-master/meta/esc50.csv"
    HUMAN_XLSX = f"{DRIVE_ROOT}/data/ESC-50-master/meta/esc50-human.xlsx"
    CACHE_DIR  = f"{DRIVE_ROOT}/cache"
    FIG_DIR, TAB_DIR = f"{DRIVE_ROOT}/figures", f"{DRIVE_ROOT}/tables"
else:
    META_CSV   = "data/ESC-50-master/meta/esc50.csv"
    HUMAN_XLSX = "data/ESC-50-master/meta/esc50-human.xlsx"
    CACHE_DIR  = "cache"
    FIG_DIR, TAB_DIR = "figures", "tables"
for d in (FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)
PREFIX = "fake_" if USE_FAKE else ""

def savefig(fig, name):
    if USE_FAKE:
        fig.text(0.5, 0.5, "DỮ LIỆU GIẢ - CHỈ ĐỂ TEST", fontsize=30, color="red",
                 alpha=0.25, ha="center", va="center", rotation=25)
    path = os.path.join(FIG_DIR, PREFIX + name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("Đã lưu:", path)

def savetab(df, name):
    path = os.path.join(TAB_DIR, PREFIX + name)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("Đã lưu:", path)

print("Chế độ:", "DỮ LIỆU GIẢ (chỉ để test)" if USE_FAKE else "DỮ LIỆU THẬT")
print("Thư mục kết quả:", RESULT_DIR)

## 1. Kiểm tra đủ kết quả (dùng cho buổi họp giữa kỳ)

Cần đủ **25 file** `exp{id}_fold{k}.json` (EXP1-EXP5 x 5 fold). Bảng dưới cho biết fold nào còn thiếu. Nếu thiếu thì các bảng sau chỉ là tạm thời.

In [ ]:
if USE_FAKE:
    ev.make_fake_results(RESULT_DIR, exps=EXP_IDS)
    print("!!! ĐANG DÙNG DỮ LIỆU GIẢ: mọi con số bên dưới KHÔNG phải kết quả thật !!!\n")

avail = ev.availability_table(EXP_IDS, RESULT_DIR)
display(avail)
missing = avail[avail["Số fold"] < 5]
if len(missing):
    print("Chưa đủ 5 fold cho:", ", ".join(missing["EXP"]), "-> kết quả bên dưới chỉ tạm thời.")

ALL = {e: ev.load_exp(e, result_dir=RESULT_DIR) for e in EXP_IDS}
ALL = {e: d for e, d in ALL.items() if d}
bad = [(e, k) for e, d in ALL.items() for k, r in d.items() if not ev.check_consistency(r)]
print("Kiểm tra khớp acc/macro_f1 với JSON:", "OK" if not bad else f"LỆCH ở {bad}")
if not ALL:
    raise SystemExit("Chưa có file kết quả nào trong " + RESULT_DIR)

## 2. Bảng kết quả tổng hợp (mục 4.3)

Mỗi EXP: mean ± std qua các fold. Baseline là EXP1, mô hình đề xuất là EXP5.

In [ ]:
df_res, per_fold = ev.summarize(list(ALL), RESULT_DIR)
show = df_res[["EXP", "Số fold", "Accuracy (%)", "Macro-F1 (%)"]]
display(show)
savetab(show, "tab_results.csv")

## 3. Trả lời RQ1-RQ3 bằng kiểm định cặp (mục 4.4)

| RQ | Cặp EXP | Giả thuyết |
|---|---|---|
| RQ1 | EXP1 vs EXP3, EXP2 vs EXP4 | H1: 128 bins cao hơn 64 bins |
| RQ2 | EXP1 vs EXP2, EXP3 vs EXP4 | H2: SpecAugment tăng Macro-F1 và giảm khoảng cách train-val |
| RQ3 | EXP4 vs EXP5 | H3: CRNN cao hơn CNN |

Bảng dùng **paired t-test theo fold** trên Macro-F1. Chỉ có 5 fold nên p-value chỉ mang tính tham khảo. Chỉ kết luận "tốt hơn" khi chênh lệch lớn hơn độ lệch chuẩn giữa các fold, nếu không viết "chưa đủ bằng chứng".

In [ ]:
df_rq = ev.rq_table(per_fold, metric=1)   # metric=1: Macro-F1 (đổi thành 0 để xem Accuracy)
display(df_rq)
savetab(df_rq, "tab_rq.csv")

In [ ]:
df_gap = ev.gap_table(list(ALL), RESULT_DIR)
print("Khoảng cách train - val (điểm % accuracy) tại epoch có val_loss thấp nhất - dùng cho H2:")
display(df_gap)
savetab(df_gap, "tab_gap_train_val.csv")

df_cost = ev.cost_table(list(ALL), RESULT_DIR)
print("Chi phí tính toán - dùng cho mục 4.5 (Thanh viết, gửi bảng này cho Thanh):")
display(df_cost)
savetab(df_cost, "tab_cost.csv")

## 4. Biểu đồ ablation có error bar

In [ ]:
colors = ["#90A4AE", "#4DB6AC", "#7986CB", "#FFB74D", "#E57373", "#A1887F"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(df_res))
for ax, (m, s, title) in zip(axes, [("acc_mean", "acc_std", "Accuracy (%)"),
                                     ("f1_mean", "f1_std", "Macro-F1 (%)")]):
    err = df_res[s].fillna(0)
    ax.bar(x, df_res[m], yerr=err, capsize=4, color=colors[:len(df_res)])
    ax.set_xticks(x)
    ax.set_xticklabels(df_res["EXP"])
    ax.set_ylabel(title)
    ax.set_ylim(0, float((df_res[m] + err).max()) * 1.25)
    ax.grid(axis="y", alpha=0.3)
    for i, v in enumerate(df_res[m]):
        ax.text(i, v + err.iloc[i] + 0.5, f"{v:.1f}", ha="center", fontsize=9)
fig.suptitle("So sánh các thí nghiệm (mean ± std qua các fold)")
savefig(fig, "fig6_ablation_bar.png")
plt.show()

## 5. Confusion matrix và F1 theo lớp (mục 4.6)

EXP dùng để phân tích lỗi được **chọn theo validation** (`val_acc_best` trung bình), không nhìn test. Có thể gán tay `BEST_EXP` ở ô tham số.

In [ ]:
if BEST_EXP is None:
    BEST_EXP, basis = ev.select_best_exp(list(ALL), RESULT_DIR)
else:
    basis = "chọn tay"
print(f"EXP dùng để phân tích lỗi: EXP{BEST_EXP}  ({basis})")
best = ALL[BEST_EXP]
classes = ev.get_classes(META_CSV)
cm = ev.confusion_pooled(best)
cmn = ev.confusion_pooled(best, normalize=True)

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(50)); ax.set_xticklabels(classes, rotation=90, fontsize=6)
ax.set_yticks(range(50)); ax.set_yticklabels(classes, fontsize=6)
ax.set_xlabel("Lớp dự đoán"); ax.set_ylabel("Lớp thật")
ax.set_title(f"Confusion matrix gộp {len(best)} fold - EXP{BEST_EXP} (chuẩn hóa theo hàng)")
fig.colorbar(im, fraction=0.046, pad=0.02)
savefig(fig, "fig6_confmat_best.png")
plt.show()

In [ ]:
f1c = ev.per_class_f1(best)
order = np.argsort(f1c)
fig, ax = plt.subplots(figsize=(8, 11))
ax.barh(range(50), f1c[order], color=[ev.GROUP_COLORS[ev.class_group(i)] for i in order])
ax.set_yticks(range(50)); ax.set_yticklabels([classes[i] for i in order], fontsize=7)
ax.set_xlabel("F1 theo lớp"); ax.set_xlim(0, 1)
ax.legend(handles=[Patch(color=c, label=n) for c, n in zip(ev.GROUP_COLORS, ev.GROUP_NAMES)],
          loc="lower right", fontsize=8)
ax.set_title(f"F1 theo lớp - EXP{BEST_EXP} (màu theo 5 nhóm âm thanh)")
savefig(fig, "fig6_f1_per_class.png")
plt.show()

grp = (pd.DataFrame({"Nhóm": [ev.GROUP_NAMES[ev.class_group(i)] for i in range(50)], "F1": f1c})
       .groupby("Nhóm")["F1"].mean().round(3).sort_values().reset_index()
       .rename(columns={"F1": "F1 trung bình"}))
display(grp)
savetab(grp, "tab_f1_theo_nhom.csv")
print("5 lớp khó nhất :", [classes[i] for i in order[:5]])
print("5 lớp dễ nhất  :", [classes[i] for i in order[::-1][:5]])

## 6. Cặp lớp hay nhầm và ví dụ file sai (mục 4.6)

Lấy 10 cặp (thật -> đoán) bị nhầm nhiều nhất, vẽ spectrogram của 2 mẫu sai chắc chắn nhất cho 5 cặp đầu. Với mỗi cặp, viết nhận xét nguyên nhân trong báo cáo (âm thanh tương đồng? nhãn lẫn bối cảnh? nhiễu nền?).

In [ ]:
pairs = ev.top_confused_pairs(cm, classes, k=10)
display(pairs[["Lớp A (thật)", "Lớp B (đoán)", "Số lần nhầm"]])
savetab(pairs, "tab_confused_pairs.csv")

In [ ]:
n_mels = next(iter(best.values()))["n_mels"]
cache_path = os.path.join(CACHE_DIR, f"logmel{n_mels}.pt")
CACHE, FN2IDX = None, {}
if (not USE_FAKE) and os.path.exists(cache_path):
    import torch
    CACHE = torch.load(cache_path, map_location="cpu")
    FN2IDX = {f: i for i, f in enumerate(CACHE["filenames"])}
    print("Đã nạp cache:", cache_path)
elif USE_FAKE:
    print("Chế độ giả: dùng spectrogram ngẫu nhiên chỉ để minh họa bố cục hình.")
else:
    print("CHƯA THẤY cache", cache_path, "-> sẽ bỏ qua hình spectrogram (kiểm tra lại CACHE_DIR / DRIVE_ROOT).")

def get_logmel(fname):
    if CACHE is not None:
        return CACHE["X"][FN2IDX[fname]][0].numpy()
    if USE_FAKE:
        rng = np.random.default_rng(zlib.crc32(fname.encode()))
        return rng.normal(size=(n_mels, 216)).cumsum(axis=0)
    return None

N_PAIRS, N_EX = 5, 2
for n, row in pairs.head(N_PAIRS).iterrows():
    ex = ev.find_error_examples(best, int(row["id_A"]), int(row["id_B"]), n=N_EX)
    k = max(len(ex), 1)
    fig, axes = plt.subplots(1, k, figsize=(5.5 * k, 3.8), squeeze=False)
    for ax, e in zip(axes[0], ex):
        S = get_logmel(e["filename"])
        if S is None:
            ax.axis("off")
            continue
        ax.imshow(S, origin="lower", aspect="auto", cmap="magma")
        ax.set_title(f"Thật: {row['Lớp A (thật)']} | Đoán: {row['Lớp B (đoán)']}\n"
                     f"{e['filename']}  (p_đoán={e['p_pred']:.2f}, p_thật={e['p_true']:.2f})", fontsize=8)
        ax.set_xlabel("Time frame"); ax.set_ylabel("Mel bin")
    fig.suptitle(f"Cặp lỗi {n + 1}: {int(row['Số lần nhầm'])} lần nhầm", fontsize=10)
    savefig(fig, f"fig6_errors_pair{n + 1}.png")
    plt.show()

## 7. (Bonus) So sánh mô hình với người nghe theo từng lớp

File `esc50-human.xlsx` (trong thư mục `meta/` của ESC-50) chứa kết quả thí nghiệm nghe của người. Mình **chưa xem được cấu trúc file này**, nên làm 2 bước:

1. Chạy ô dưới để in các sheet và vài dòng đầu.
2. Điền `HUMAN_SHEET`, `HUMAN_CLASS_COL`, `HUMAN_ACC_COL` (tên sheet, tên cột chứa tên lớp, tên cột chứa độ chính xác của người theo lớp) rồi chạy ô tiếp theo. Nếu file ghi theo từng clip thì nhóm theo lớp và lấy trung bình trước (xem ghi chú trong ô).

In [ ]:
HUMAN_SHEET = 0            # tên hoặc số thứ tự sheet
HUMAN_CLASS_COL = None     # ví dụ: "category"
HUMAN_ACC_COL = None       # ví dụ: "accuracy"

xl = None
if os.path.exists(HUMAN_XLSX):
    xl = pd.ExcelFile(HUMAN_XLSX)
    print("Các sheet:", xl.sheet_names)
    for s in xl.sheet_names:
        d = xl.parse(s)
        print(f"--- sheet '{s}': {d.shape}")
        display(d.head(5))
else:
    print("Không thấy", HUMAN_XLSX, "-> bỏ qua phần bonus này.")

In [ ]:
if xl is not None and HUMAN_CLASS_COL and HUMAN_ACC_COL:
    hum = xl.parse(HUMAN_SHEET)
    norm = lambda s: str(s).strip().lower().replace(" ", "_")
    # Nếu file ghi theo từng clip: nhóm theo lớp và lấy trung bình
    hum = hum.groupby(HUMAN_CLASS_COL)[HUMAN_ACC_COL].mean().reset_index()
    if hum[HUMAN_ACC_COL].max() > 1.0:      # đang ở dạng phần trăm
        hum[HUMAN_ACC_COL] = hum[HUMAN_ACC_COL] / 100.0
    hum["key"] = hum[HUMAN_CLASS_COL].map(norm)
    macc = ev.per_class_acc(best)
    mod = pd.DataFrame({"key": [norm(c) for c in classes], "Lớp": classes, "Mô hình": macc})
    comp = mod.merge(hum[["key", HUMAN_ACC_COL]], on="key", how="inner").rename(columns={HUMAN_ACC_COL: "Người nghe"})
    print(f"Khớp được {len(comp)}/50 lớp.")
    comp["Chênh (mô hình - người)"] = (comp["Mô hình"] - comp["Người nghe"]).round(3)
    display(comp.sort_values("Chênh (mô hình - người)").drop(columns="key"))
    savetab(comp.drop(columns="key"), "tab_model_vs_human.csv")

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(comp["Người nghe"], comp["Mô hình"], s=25)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    for _, r in comp.iterrows():
        if abs(r["Chênh (mô hình - người)"]) > 0.25:
            ax.annotate(r["Lớp"], (r["Người nghe"], r["Mô hình"]), fontsize=7)
    ax.set_xlabel("Độ chính xác người nghe"); ax.set_ylabel(f"Độ chính xác EXP{BEST_EXP}")
    ax.set_title("Mô hình so với người nghe theo từng lớp")
    savefig(fig, "fig6_model_vs_human.png")
    plt.show()
else:
    print("Bỏ qua: chưa điền HUMAN_CLASS_COL / HUMAN_ACC_COL (hoặc chưa có file).")

## 8. Danh sách clip để nghe lại (bonus)

20 clip mô hình đoán **sai nhưng rất tự tin**. Nghe từng clip (file `audio/` trong ESC-50), điền 2 cột cuối trong `tab_nghe_lai.csv`:
- **kết luận:** `mô hình` (clip rõ ràng, mô hình sai) hoặc `dữ liệu` (nhãn lẫn bối cảnh, nhiều âm thanh cùng lúc, nhãn đáng ngờ),
- **ghi chú khi nghe:** nghe thấy gì.

Kết quả đưa vào mục 4.6 (tỉ lệ lỗi do dữ liệu so với lỗi do mô hình) và 4.8 (thảo luận).

In [ ]:
ears = ev.most_confident_errors(best, classes, n=20)
display(ears)
savetab(ears, "tab_nghe_lai.csv")

## 9. Tổng kết các file đã tạo

In [ ]:
print("Hình   :", sorted(f for f in os.listdir(FIG_DIR) if f.startswith(PREFIX)))
print("Bảng   :", sorted(f for f in os.listdir(TAB_DIR) if f.startswith(PREFIX)))
if USE_FAKE:
    print("\nĐây là file chạy thử bằng dữ liệu giả (tiền tố fake_). Khi có kết quả thật: "
          "đặt USE_FAKE = False, chạy lại, rồi xóa các file fake_ trước khi commit.")
else:
    print("\nCopy figures/ và tables/ vào repo, commit lên nhánh an-eval và mở Pull Request.")